# Speed Bump Detector: YOLO11 training on Colab

**Before you start:** `Runtime → Change runtime type → T4 GPU`.
Then upload `speed_bump_yolo.zip` to Google Drive in a folder named **`speed-bump`**: `MyDrive/speed-bump/speed_bump_yolo.zip`.
Run the cells top to bottom. Results (weights, plots, exports) are saved to `MyDrive/speed-bump/runs/`.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install Ultralytics (version pinned so results are reproducible)

In [ ]:
!pip -q install ultralytics==8.3.217
import ultralytics; ultralytics.checks()

## 3. Mount Google Drive and unpack the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/speed-bump'   # change if you used another folder
ZIP = f'{DRIVE_DIR}/speed_bump_yolo.zip'
RUNS = f'{DRIVE_DIR}/runs'

import os
assert os.path.exists(ZIP), f'Upload speed_bump_yolo.zip to {DRIVE_DIR} first'
!rm -rf /content/data && mkdir -p /content/data && unzip -q "{ZIP}" -d /content/data

# point the dataset config at the unzipped folder
DATA_YAML = '/content/data/yolo/bumps.yaml'
txt = open(DATA_YAML).read().replace('path: .', 'path: /content/data/yolo')
open(DATA_YAML, 'w').write(txt)
print(txt)

## 4. Sanity check: counts per split and class

In [ ]:
from pathlib import Path
from collections import Counter
root = Path('/content/data/yolo')
for split in ['train', 'val', 'test']:
    lbls = list((root / 'labels' / split).glob('*.txt'))
    boxes = Counter()
    empty = 0
    for p in lbls:
        lines = [l for l in p.read_text().splitlines() if l.strip()]
        empty += not lines
        boxes.update(l.split()[0] for l in lines)
    print(f"{split:5s}: {len(lbls)} images | marked boxes {boxes['0']} | unmarked boxes {boxes['1']} | background images {empty}")

## 5. Train

* `yolo11n`: the smallest model, fast enough for phones. Try `yolo11s.pt` afterwards and compare.
* `flipud=0`: the road is always at the bottom of the frame, so never flip vertically.
* About 15–25 min on a T4. If Colab disconnects, run the **resume** cell below.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')            # COCO-pretrained starting point
results = model.train(
    data=DATA_YAML,
    epochs=100,
    patience=30,                      # stop early if val mAP stops improving
    imgsz=640,
    batch=16,
    flipud=0.0, fliplr=0.5,
    degrees=3, perspective=0.0005,    # small camera-tilt augmentation
    project=RUNS, name='yolo11n_v1',
    exist_ok=True,
    seed=0,
)

**Only if the session dropped mid-training:** resume from the last checkpoint saved on Drive.

In [ ]:
# from ultralytics import YOLO
# YOLO(f'{RUNS}/yolo11n_v1/weights/last.pt').train(resume=True)

## 6. Evaluate on the held-out TEST split (never used during training)

In [ ]:
best = YOLO(f'{RUNS}/yolo11n_v1/weights/best.pt')
m = best.val(data=DATA_YAML, split='test', imgsz=640, project=RUNS, name='yolo11n_v1_test', exist_ok=True)

print(); print(f"mAP50 overall: {m.box.map50:.3f}   mAP50-95: {m.box.map:.3f}")
for i, c in enumerate(m.ap_class_index):
    print(f"  {best.names[c]:14s} precision {m.box.p[i]:.2f}  recall {m.box.r[i]:.2f}  mAP50 {m.box.ap50[i]:.2f}")
print(); print('Note: the test split has only 4 unmarked images, so treat unmarked numbers as very rough.')

## 7. False alarms: how many bump-free road images trigger a detection?

In [ ]:
bg = [p for p in (root / 'images' / 'test').glob('rdd_*.jpg')]
preds = best.predict([str(p) for p in bg], conf=0.25, imgsz=640, verbose=False, stream=True)
fp = [p.path for p in preds if len(p.boxes)]
print(f'{len(fp)} / {len(bg)} background test images got a false detection ({100*len(fp)/len(bg):.1f}%)')

## 8. Look at predictions on test images

In [ ]:
import random
from IPython.display import Image, display
random.seed(1)
bumps = [p for p in (root / 'images' / 'test').glob('*.jpg') if not p.name.startswith('rdd_')]
for r in best.predict([str(p) for p in random.sample(bumps, 6)], conf=0.25, imgsz=640,
                      save=True, project=RUNS, name='yolo11n_v1_preview', exist_ok=True, verbose=False):
    pass
for f in sorted(Path(f'{RUNS}/yolo11n_v1_preview').glob('*.jpg'))[:6]:
    display(Image(filename=str(f), width=420))

Training curves and confusion matrix:

In [ ]:
for f in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png']:
    p = Path(f'{RUNS}/yolo11n_v1/{f}')
    if p.exists(): display(Image(filename=str(p), width=800))

## 9. Try it on your own video (optional)
Upload a dashcam-style `.mp4` to `MyDrive/speed-bump/`, set the name below, and run. The annotated video is saved to `runs/video/`.

In [ ]:
VIDEO = f'{DRIVE_DIR}/test_drive.mp4'     # <- your file
if os.path.exists(VIDEO):
    best.predict(VIDEO, conf=0.35, imgsz=640, save=True, project=RUNS, name='video', exist_ok=True, verbose=False)
    print('saved to', f'{RUNS}/video')
else:
    print('no video found, skipping')

## 10. Export for deployment

In [ ]:
best.export(format='onnx', imgsz=640, simplify=True)        # laptop / desktop (ONNX Runtime)
# best.export(format='tflite', imgsz=640, int8=True, data=DATA_YAML)  # Android (slower export)
!ls -lh {RUNS}/yolo11n_v1/weights/

Done. `best.pt` and `best.onnx` are in `MyDrive/speed-bump/runs/yolo11n_v1/weights/`.
Write the test numbers from step 6 and the false-alarm rate from step 7 into the run log (see `docs/COLAB_TRAINING.md`).